# Packoscope · Entrega 2

O projeto visa modelar um Intrusion Detection System (IDS) a partir do dataset CIC-IDS-2017, gerado sinteticamente pelo Canadian Institute for Cybersecurity, Universidade de Brunswick, Canadá. Este notebook contém o experimento da Entrega 2: a construção de um pipeline parametrizável de pré-processamento e a avaliação de todas as combinações de técnicas com o kNN (k = 7), em validação cruzada 5-fold. A análise exploratória da Entrega 1 está em `eda.ipynb`.

O espaço tem 72 combinações (2 × 4 × 3 × 3). A etapa de encoding foi omitida, porque todos os atributos que entram no modelo são numéricos: `Label` vira o alvo binário, e `source_file` e `Destination Port` ficam de fora, como decidido na Entrega 1.

In [ ]:
# imports necessarios para o experimento
import platform
import time
from itertools import product
from pathlib import Path

import imblearn
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import fbeta_score, make_scorer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

In [ ]:
# configs de estilo
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

## Configuração do Experimento

Os parâmetros abaixo são fixos para todas as combinações. `N_AMOSTRA` define o tamanho da amostra (ver a seção de amostragem) e `METRICA` é a métrica usada para ordenar e comparar os resultados. Foi escolhido o F2, que dá mais peso ao recall do que à precisão, seguindo a prioridade definida na Entrega 1: deixar passar um ataque custa mais do que um falso alarme. Os resultados são gravados em `resultados/`, um arquivo por tamanho de amostra.

In [ ]:
# parametros do experimento
SEED = 42
N_AMOSTRA = 300_000
K_VIZINHOS = 7
N_FOLDS = 5
METRICA = "f2"

# um arquivo por tamanho de amostra, pra nao misturar rodadas diferentes
ARQ_RESULTADOS = Path("resultados") / f"pipelines_n{N_AMOSTRA}.csv"

As versões das bibliotecas são registradas para o protocolo do relatório.

In [ ]:
# versoes usadas no experimento
versoes = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "imbalanced-learn": imblearn.__version__,
}
pd.Series(versoes, name="versao")

## Preparação Fixa

Os passos desta seção valem para todas as combinações e não fazem parte do espaço de pipelines. Primeiro, saem da base o metadado (`source_file` e `Destination Port`) e a segunda coluna de cada um dos 7 pares idênticos encontrados na EDA. Antes de descartar, é conferido que as colunas continuam idênticas. Em seguida, os valores `inf` e os negativos impossíveis (`Flow Duration` e `Fwd Header Length` menores que zero) viram `NaN`, para serem tratados pela etapa de imputação. Essa troca é feita linha a linha e não aprende nada dos dados, então pode ser feita antes do split sem causar vazamento.

Por último, foram removidas as linhas duplicadas. A remoção é feita no espaço que o modelo vê, ou seja, depois de retirar as colunas e considerando o alvo binário: duas linhas que diferem só em `Destination Port` viram cópias para o kNN. Se uma cópia fica no treino e a outra no teste, o kNN encontra um vizinho a distância zero e a métrica fica otimista. Por isso as duplicatas são removidas antes da amostragem e da divisão em folds.

In [ ]:
# carregar dados
df = pd.read_parquet("data/pacotes.parquet")

# segunda coluna de cada par identico linha a linha (ver eda.ipynb)
COPIAS = {
    "Subflow Bwd Packets": "Total Backward Packets",
    "SYN Flag Count": "Fwd PSH Flags",
    "Avg Fwd Segment Size": "Fwd Packet Length Mean",
    "CWE Flag Count": "Fwd URG Flags",
    "Fwd Header Length.1": "Fwd Header Length",
    "Subflow Fwd Packets": "Total Fwd Packets",
    "Avg Bwd Segment Size": "Bwd Packet Length Mean",
}

# garante que continuam identicas antes de descartar
for copia, original in COPIAS.items():
    a = df[copia].to_numpy(dtype="float64")
    b = df[original].to_numpy(dtype="float64")
    assert np.array_equal(a, b, equal_nan=True), copia

# metadado e copias ficam fora do modelo
df = df.drop(columns=["source_file", "Destination Port", *COPIAS])
ATRIBUTOS = df.columns.drop("Label")

# inf e negativos impossiveis viram nan; troca linha a linha, nao aprende nada dos dados
# so as colunas que mudam viram float64, a conversao da base toda fica pra amostra (memoria)
decimais = df[ATRIBUTOS].select_dtypes("float").columns
df[decimais] = df[decimais].replace([np.inf, -np.inf], np.nan)
for col in ["Flow Duration", "Fwd Header Length"]:
    df[col] = df[col].astype("float64").mask(df[col] < 0)

# alvo binario: malicioso = 1 (classe positiva)
df["malicioso"] = (df["Label"] != "BENIGN").astype("int8")

# duplicata no espaco que o modelo ve (atributos + alvo), antes da amostra e do split
n_antes = len(df)
df = df.drop_duplicates(subset=[*ATRIBUTOS, "malicioso"]).reset_index(drop=True)
print(f"linhas: {n_antes:,} -> {len(df):,} ({n_antes - len(df):,} duplicatas removidas)")
print(f"atributos: {len(ATRIBUTOS)} | maliciosos: {df['malicioso'].mean():.2%}")

### Amostragem Estratificada

Rodar o kNN na base inteira levaria cerca de 40 horas para as 72 combinações, segundo o piloto de tempo. Por isso foi utilizada uma amostra aleatória estratificada pela `Label` original: cada uma das 15 classes é sorteada na mesma fração. Dessa forma, a proporção de maliciosos e a composição dos tipos de ataque ficam iguais às da base. A amostra não é balanceada, já que o balanceamento é uma das etapas avaliadas. As classes raras ficam com poucas linhas (Heartbleed com cerca de 1), o que não afeta o alvo binário.

In [ ]:
# mesma fracao em cada classe, mantem a proporcao real (sem balancear aqui)
frac = N_AMOSTRA / len(df)
amostra = (
    df.groupby("Label", observed=True)
    .sample(frac=frac, random_state=SEED)
    .reset_index(drop=True)
)

# confere se a composicao da amostra bate com a da base
comparacao = pd.DataFrame({
    "base_%": df["Label"].value_counts(normalize=True).mul(100),
    "amostra_%": amostra["Label"].value_counts(normalize=True).mul(100),
    "amostra_n": amostra["Label"].value_counts(),
}).round(3)

print(f"amostra: {len(amostra):,} linhas (fracao {frac:.4f})")
print(
    f"maliciosos: base {df['malicioso'].mean():.2%}"
    f" | amostra {amostra['malicioso'].mean():.2%}"
)
comparacao

In [ ]:
# a base inteira nao e mais usada, libera memoria
del df

### Alvo e Atributos

O alvo é o `malicioso` criado na preparação, com malicioso = 1 como classe positiva. Os 70 atributos restantes formam `X`. As 8 colunas constantes continuam aqui e são removidas dentro do pipeline por `VarianceThreshold`, que é ajustado só no treino de cada fold.

In [ ]:
# alvo e atributos da amostra
y = amostra["malicioso"].to_numpy()
X = amostra[ATRIBUTOS].astype("float64")

print(f"X: {X.shape[0]:,} x {X.shape[1]} | linhas com nan: {int(X.isna().any(axis=1).sum()):,}")
print(f"maliciosos: {y.mean():.2%}")

### Folds

Os 5 folds são gerados uma única vez, estratificados pelo alvo, e a mesma lista é passada para todas as combinações. Assim, as diferenças entre pipelines não vêm de divisões diferentes dos dados.

In [ ]:
# folds gerados uma vez e reutilizados por todos os pipelines
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X, y))

# proporcao de maliciosos no teste de cada fold
[f"{y[teste].mean():.2%}" for _, teste in FOLDS]

### Verificação de Duplicatas entre Treino e Teste

Para confirmar que a remoção de duplicatas funcionou, foi conferido em cada fold se alguma linha do teste tem uma cópia exata (mesmos atributos e mesmo alvo) no treino. O `assert` interrompe o notebook se isso acontecer.

In [ ]:
# nenhuma linha do teste pode ter copia exata (atributos e alvo) no treino do mesmo fold
chave = pd.util.hash_pandas_object(amostra[[*ATRIBUTOS, "malicioso"]], index=False).to_numpy()
for i, (treino, teste) in enumerate(FOLDS):
    no_treino = int(np.isin(chave[teste], chave[treino]).sum())
    print(f"fold {i}: {len(treino):,} treino | {len(teste):,} teste | copias do teste no treino: {no_treino}")
    assert no_treino == 0

## Espaço de Pipelines

| Etapa | Opções | Justificativa (EDA) |
|---|---|---|
| Imputação | A1: mediana · A2: constante 0 + indicador | 0,10% das linhas têm `NaN` ou `inf`, todas com duração zero. 69,88% das linhas com `NaN` em `Flow Bytes/s` são maliciosas, por isso a A2 guarda essa informação em uma coluna indicadora. |
| Normalização | N0: sem · N1: `StandardScaler` · N2: `MinMaxScaler` · N3: `RobustScaler` | As escalas vão de 0/1 (flags) até 10⁹ (`Flow Bytes/s`), e 84,8% dos fluxos são outlier em pelo menos um atributo. |
| Balanceamento | B0: sem · B1: `RandomUnderSampler` · B2: `SMOTE` | 15,30% de maliciosos depois de remover as duplicatas. |
| Redução | R0: sem · R1: `PCA` (95% da variância) · R2: `SelectKBest(f_classif, k=20)` | Vários pares com \|r\| > 0,9 e separabilidade individual máxima de 0,74. |

A baseline (P01) usa a imputação pela mediana e nenhuma etapa opcional. Ela ainda tem imputação porque o kNN não aceita `NaN`. O valor `k = 20` do `SelectKBest` corresponde a cerca de um terço dos 62 atributos e foi fixado antes de rodar, assim como os 95% de variância do PCA, para não criar combinações fora do espaço.

In [ ]:
# opcoes de cada etapa; None = etapa nao aplicada
OPCOES = {
    "imputacao": {
        "A1": SimpleImputer(strategy="median"),
        "A2": SimpleImputer(strategy="constant", fill_value=0, add_indicator=True),
    },
    "normalizacao": {
        "N0": None,
        "N1": StandardScaler(),
        "N2": MinMaxScaler(),
        "N3": RobustScaler(),
    },
    "balanceamento": {
        "B0": None,
        "B1": RandomUnderSampler(random_state=SEED),
        "B2": SMOTE(random_state=SEED),
    },
    "reducao": {
        "R0": None,
        "R1": PCA(n_components=0.95, random_state=SEED),
        "R2": SelectKBest(f_classif, k=20),
    },
}
ETAPAS = list(OPCOES)

# todas as combinacoes, P01 e a baseline
combinacoes = pd.DataFrame(list(product(*(OPCOES[e] for e in ETAPAS))), columns=ETAPAS)
combinacoes.insert(0, "id", [f"P{i:02d}" for i in range(1, len(combinacoes) + 1)])

print(f"{len(combinacoes)} combinacoes")
combinacoes.head()

### Montagem do Pipeline

A função abaixo monta o pipeline de uma combinação. A ordem dentro de cada fold é: imputação, remoção de constantes, normalização, redução e balanceamento, seguidos do kNN. A normalização vem antes da redução porque o PCA depende da escala, e o balanceamento vem por último porque o SMOTE calcula distâncias e fica mais barato em menos dimensões.

Foi utilizado o `Pipeline` do imbalanced-learn, que aceita samplers e aplica o balanceamento só no treino de cada fold. O fold de teste mantém a proporção real. Etapas não aplicadas são retiradas do pipeline. O exemplo mostra a última combinação (P72), que tem todas as etapas.

In [ ]:
def montar_pipeline(linha):
    # ordem dentro do fold: imputacao, constantes, normalizacao, reducao, balanceamento, knn
    passos = [
        ("imputacao", OPCOES["imputacao"][linha["imputacao"]]),
        ("constantes", VarianceThreshold()),
        ("normalizacao", OPCOES["normalizacao"][linha["normalizacao"]]),
        ("reducao", OPCOES["reducao"][linha["reducao"]]),
        ("balanceamento", OPCOES["balanceamento"][linha["balanceamento"]]),
        ("knn", KNeighborsClassifier(n_neighbors=K_VIZINHOS, n_jobs=-1)),
    ]
    # etapa nao aplicada sai; clone pra nenhum pipeline reaproveitar objeto ja ajustado
    return Pipeline([(nome, clone(est)) for nome, est in passos if est is not None])


montar_pipeline(combinacoes.iloc[-1])

### Métricas

Foram calculadas acurácia, precisão, recall, F1, F2 e AUC, todas com malicioso como classe positiva. A AUC usa a probabilidade do kNN, que com k = 7 só assume 8 valores (de 0/7 a 7/7), então a curva ROC tem poucos pontos.

In [ ]:
# malicioso = 1, entao precisao, recall e f sao da classe maliciosa
METRICAS = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "f2": make_scorer(fbeta_score, beta=2),
    "roc_auc": "roc_auc",
}

# colunas fixas do arquivo de resultados, mesmo quando a combinacao falha
COLUNAS = [
    "id", *ETAPAS,
    *[f"{m}_{s}" for m in METRICAS for s in ("media", "dp")],
    "fit_s", "predicao_s", "total_s", "erro",
]

## Execução

O loop roda o 5-fold de cada combinação e grava uma linha no arquivo de resultados logo em seguida, com média e desvio-padrão de cada métrica, tempo médio de fit e de predição por fold, tempo total e a mensagem de erro quando a combinação falha. Uma falha não interrompe as demais.

Se o notebook for interrompido, basta rodar a célula de novo: as combinações já gravadas são puladas. Para refazer do zero, é preciso apagar o arquivo de resultados.

In [ ]:
def avaliar(linha):
    # 5-fold de uma combinacao, resumido em media e desvio de cada metrica
    inicio = time.perf_counter()
    r = cross_validate(
        montar_pipeline(linha), X, y, cv=FOLDS, scoring=METRICAS, error_score="raise"
    )
    resumo = {}
    for m in METRICAS:
        resumo[f"{m}_media"] = r[f"test_{m}"].mean()
        resumo[f"{m}_dp"] = r[f"test_{m}"].std()
    resumo["fit_s"] = r["fit_time"].mean()
    resumo["predicao_s"] = r["score_time"].mean()
    resumo["total_s"] = time.perf_counter() - inicio
    return resumo

In [ ]:
# retoma de onde parou: combinacao ja gravada nao roda de novo
ARQ_RESULTADOS.parent.mkdir(exist_ok=True)
feitos = set(pd.read_csv(ARQ_RESULTADOS)["id"]) if ARQ_RESULTADOS.exists() else set()
print(f"ja gravadas: {len(feitos)} de {len(combinacoes)}")

for _, linha in combinacoes.iterrows():
    if linha["id"] in feitos:
        continue
    try:
        resumo, erro = avaliar(linha), ""
    except Exception as e:  # noqa: BLE001
        # falha nao derruba o loop, a mensagem vai para a tabela
        resumo, erro = {}, f"{type(e).__name__}: {e}"

    # grava a cada combinacao pra nao perder o que ja rodou
    registro = {**linha.to_dict(), **resumo, "erro": erro}
    pd.DataFrame([registro], columns=COLUNAS).to_csv(
        ARQ_RESULTADOS, mode="a", header=not ARQ_RESULTADOS.exists(), index=False
    )

    status = erro or f"{METRICA} = {resumo[f'{METRICA}_media']:.4f}"
    print(f"{linha['id']}  {status}  ({resumo.get('total_s', 0):.0f} s)")

## Resultados

### Tabela Completa

A tabela abaixo tem uma linha por combinação, ordenada pela métrica principal, e é a base do anexo do relatório.

In [ ]:
# tabela completa, base do anexo do relatorio
res = pd.read_csv(ARQ_RESULTADOS)
media, dp = f"{METRICA}_media", f"{METRICA}_dp"

print(f"{len(res)} combinacoes | falhas: {int(res['erro'].notna().sum())}")
res.sort_values(media, ascending=False)

### Comparação com a Baseline

Cada combinação foi comparada com a baseline (P01) pela métrica principal. Foi considerada empate toda diferença menor que o maior dos dois desvios-padrão entre folds. A tabela mostra as 5 melhores e as 5 piores combinações.

In [ ]:
base = res.loc[res["id"] == "P01"].iloc[0]

# empate: diferenca menor que o maior dos dois desvios
res["delta_baseline"] = res[media] - base[media]
limite = np.maximum(res[dp], base[dp])
res["vs_baseline"] = np.select(
    [res["delta_baseline"].abs() < limite, res["delta_baseline"] > 0],
    ["empate", "melhor"],
    default="pior",
)
print(res["vs_baseline"].value_counts().to_string())

# quantas combinacoes empatam com a melhor
ranking = res.sort_values(media, ascending=False)
melhor = ranking.iloc[0]
empatam = (melhor[media] - res[media]) < np.maximum(res[dp], melhor[dp])
print(f"\nmelhor: {melhor['id']} | empatam com ela: {int(empatam.sum()) - 1}")

cols = ["id", *ETAPAS, media, dp, "delta_baseline", "vs_baseline"]
pd.concat([ranking.head(5), ranking.tail(5)])[cols]

### Efeito de Cada Etapa

Para isolar o efeito de uma etapa, as 72 combinações foram agrupadas pela opção adotada nela. Cada opção de normalização, por exemplo, aparece em 18 combinações, que cobrem todas as opções das outras etapas.

In [ ]:
# cada grupo cobre todas as opcoes das outras etapas, entao isola o efeito da etapa
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, etapa in zip(axes.flat, ETAPAS):
    sns.boxplot(data=res, x=etapa, y=media, order=list(OPCOES[etapa]), ax=ax)
    ax.set_title(etapa)
    ax.set_xlabel("")
fig.suptitle(f"{METRICA} por opção de cada etapa (média dos 5 folds)")
plt.tight_layout()
plt.show()

# resumo numerico de cada grupo
pd.concat({e: res.groupby(e)[media].agg(["mean", "std", "min", "max"]) for e in ETAPAS})

### Interações entre Etapas

Os heatmaps cruzam duas etapas e mostram a média da métrica principal. Se o efeito de uma técnica muda conforme a opção da outra etapa, há uma interação. O par normalização × redução é o primeiro a observar, porque o PCA depende da escala.

In [ ]:
# efeito que muda de uma linha para outra indica interacao
pares = [("normalizacao", "reducao"), ("normalizacao", "balanceamento"), ("reducao", "balanceamento")]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, (a, b) in zip(axes, pares):
    tabela = res.pivot_table(index=a, columns=b, values=media, aggfunc="mean")
    sns.heatmap(tabela, annot=True, fmt=".3f", cmap="viridis", ax=ax)
    ax.set_title(f"{a} × {b}")
plt.tight_layout()
plt.show()

### Precisão e Recall por Balanceamento

O balanceamento deve aumentar o recall e diminuir a precisão. Por isso as duas métricas são olhadas separadas, além do F1 e do F2.

In [ ]:
# media de cada metrica por opcao de balanceamento
res.groupby("balanceamento")[["precision_media", "recall_media", "f1_media", "f2_media"]].mean()

### Custo

No kNN o fit só guarda os dados de treino, e o custo fica na predição, que compara cada ponto de teste com todo o treino. A primeira tabela mostra o tempo médio por opção de cada etapa e a segunda, as 5 combinações mais caras.

In [ ]:
# tempo medio por opcao de cada etapa
pd.concat({e: res.groupby(e)[["fit_s", "predicao_s", "total_s"]].mean() for e in ETAPAS}).round(2)

In [ ]:
# combinacoes mais caras
res.sort_values("total_s", ascending=False)[["id", *ETAPAS, "fit_s", "predicao_s", "total_s"]].head(5)

## Diagnósticos Complementares

As células abaixo não fazem parte da validação cruzada. Elas usam a amostra inteira para ajudar a explicar os resultados.

### RobustScaler e Colunas com IQR Zero

Quando o IQR de uma coluna é zero, o `RobustScaler` usa escala 1, ou seja, a coluna não é reescalada. A tabela compara o maior valor absoluto dessas colunas depois do `RobustScaler` e do `StandardScaler`, com `Flow Duration` como referência.

In [ ]:
# imputa e tira constantes, igual ao comeco do pipeline
Xi = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(X), columns=X.columns)
Xi = Xi.loc[:, Xi.nunique() > 1]

# colunas com iqr zero ficam com escala 1 no robustscaler
iqr = Xi.quantile(0.75) - Xi.quantile(0.25)
sem_escala = iqr.index[iqr == 0].tolist()

diag = pd.DataFrame({
    "iqr": iqr,
    "max_abs_robust": np.abs(RobustScaler().fit_transform(Xi)).max(axis=0),
    "max_abs_standard": np.abs(StandardScaler().fit_transform(Xi)).max(axis=0),
}, index=Xi.columns)

print(f"colunas com iqr = 0: {len(sem_escala)}")
diag.loc[sem_escala + ["Flow Duration"]]

### Componentes do PCA por Normalização

Foi contado quantos componentes o PCA mantém para explicar 95% da variância com cada opção de normalização. Sem normalização, a variância fica concentrada nas colunas de maior escala.

In [ ]:
# quantos componentes o pca guarda com cada normalizacao
for opcao, scaler in OPCOES["normalizacao"].items():
    Z = Xi if scaler is None else clone(scaler).fit_transform(Xi)
    n = PCA(n_components=0.95, random_state=SEED).fit(Z).n_components_
    print(f"{opcao}  {type(scaler).__name__ if scaler else 'sem normalizacao':<18} {n} componentes")

### Teste de Permutação

Se nenhuma etapa usa informação do fold de teste, um pipeline treinado com os rótulos embaralhados não consegue prever nada, e a AUC fica perto de 0,5. Foram testadas a baseline (P01) e duas combinações com as etapas que usam o alvo no ajuste: P18 (`SelectKBest` com `SMOTE`) e P72 (todas as etapas). Para reduzir o tempo, foi usada uma subamostra de 40.000 linhas.

In [ ]:
# rotulos embaralhados: sem vazamento, a auc fica perto de 0,5
rng = np.random.default_rng(SEED)
sub = rng.choice(len(X), min(40_000, len(X)), replace=False)
y_perm = rng.permutation(y[sub])
folds_perm = list(
    StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(X.iloc[sub], y_perm)
)

perm = []
for pid in ["P01", "P18", "P72"]:
    linha = combinacoes.set_index("id").loc[pid]
    r = cross_validate(montar_pipeline(linha), X.iloc[sub], y_perm, cv=folds_perm, scoring="roc_auc")
    perm.append({"id": pid, **linha.to_dict(), "auc_media": r["test_score"].mean(), "auc_dp": r["test_score"].std()})

pd.DataFrame(perm).round(3)